# Model Training & Evaluation

This notebook walks through the same pipeline as `src/model_training.py`
interactively: preprocessing, training three algorithms with cross-validated
hyperparameter search, and comparing their performance.

Run `python src/model_training.py` from the project root to regenerate
`models/house_price_model.pkl` and `docs/model_performance.json` as files —
this notebook is for inspection and narrative, not for producing the
production artifact.

In [1]:
import sys, json
sys.path.append('../src')
import pandas as pd
from data_preprocessing import prepare_dataset, get_feature_lists
from model_training import train_and_compare

df = prepare_dataset('../data/house_prices.csv')
df.head()

2026-09-10 04:50:38,274 [INFO] Loaded raw data: 300 rows, 8 columns


2026-09-10 04:50:38,278 [INFO] Dropped 0 duplicate rows


2026-09-10 04:50:38,282 [INFO] Dropped 0 rows with impossible values


2026-09-10 04:50:38,284 [INFO] Dropped 0 rows with unrecognised category values


,Property_ID,Area,Bedrooms,Bathrooms,Age,Location,Property_Type,Price,total_rooms,bath_bed_ratio,is_new,area_per_room,age_bucket
0,PROP0001,3712,4,3,36,Rural,House,22260000,7,0.75,0,530.285714,30+
1,PROP0002,1591,4,1,35,Suburb,House,16057500,5,0.25,0,318.200000,30+
2,PROP0003,1646,4,3,20,Rural,Villa,12730000,7,0.75,0,235.142857,16-30
3,PROP0004,4814,1,2,13,City Center,Villa,50840000,3,2.00,0,1604.666667,6-15
4,PROP0005,800,4,2,38,Suburb,Apartment,10650000,6,0.50,0,133.333333,30+


## Feature set used for modeling

Numeric and categorical features (raw + engineered) fed into the preprocessing `ColumnTransformer`:

In [2]:
numeric_features, categorical_features = get_feature_lists()
print('Numeric:', numeric_features)
print('Categorical:', categorical_features)

Numeric: ['Area', 'Bedrooms', 'Bathrooms', 'Age', 'total_rooms', 'bath_bed_ratio', 'is_new', 'area_per_room']
Categorical: ['Location', 'Property_Type', 'age_bucket']


## Train and compare all three algorithms

This runs `GridSearchCV` for Random Forest and Gradient Boosting, and a plain fit for Linear Regression, each evaluated with 5-fold cross-validation and a held-out 20% test set.

In [3]:
results = train_and_compare(csv_path='../data/house_prices.csv')
print(json.dumps(results['results'], indent=2))

2026-09-10 04:50:38,315 [INFO] Loaded raw data: 300 rows, 8 columns


2026-09-10 04:50:38,317 [INFO] Dropped 0 duplicate rows


2026-09-10 04:50:38,320 [INFO] Dropped 0 rows with impossible values


2026-09-10 04:50:38,322 [INFO] Dropped 0 rows with unrecognised category values


2026-09-10 04:50:38,328 [INFO] Train size: 240, Test size: 60


...

2026-09-10 04:50:55,758 [INFO] Training gradient_boosting ...


2026-09-10 04:51:02,478 [INFO] gradient_boosting -> MAE=812105  R2=0.990  CV_R2=0.989+-0.003


2026-09-10 04:51:02,480 [INFO] Best model: gradient_boosting (R2=0.990)


2026-09-10 04:51:02,487 [INFO] Saved best model to /home/claude/house-price-ml/notebooks/../models/house_price_model.pkl


{
  "linear_regression": {
    "mae": 2285844.6427080124,
    "rmse": 3020306.139891131,
    "r2": 0.9359472671058938,
    "mape": 13.87598492985955,
    "cv_r2_mean": 0.9535291916199696,
    "cv_r2_std": 0.007140261991125047,
    "best_params": {},
    "train_time_seconds": 0.07
  },
  "random_forest": {
    "mae": 1389016.6666666667,
    "rmse": 1890174.8840495332,
    "r2": 0.9749135210563797,
    "mape": 6.270568183424811,
    "cv_r2_mean": 0.9707236070182879,
    "cv_r2_std": 0.004755975047237572,
    "best_params": {
      "regressor__max_depth": null,
      "regressor__min_samples_leaf": 1,
      "regressor__n_estimators": 100
    },
    "train_time_seconds": 17.35
  },
  "gradient_boosting": {
    "mae": 812104.6169268097,
    "rmse": 1199574.0261261608,
    "r2": 0.9898960813624921,
    "mape": 3.9380385051330493,
    "cv_r2_mean": 0.9889513239672688,
    "cv_r2_std": 0.0029904581215209483,
    "best_params": {
      "regressor__learning_rate": 0.1,
      "regressor__max_depth

In [4]:
import pandas as pd
comparison = pd.DataFrame(results['results']).T[['mae','rmse','r2','mape','cv_r2_mean','cv_r2_std']]
comparison.sort_values('r2', ascending=False)

,mae,rmse,r2,mape,cv_r2_mean,cv_r2_std
gradient_boosting,812104.616927,1199574.026126,0.989896,3.938039,0.988951,0.00299
random_forest,1389016.666667,1890174.88405,0.974914,6.270568,0.970724,0.004756
linear_regression,2285844.642708,3020306.139891,0.935947,13.875985,0.953529,0.00714


## Interpretation

- **Gradient Boosting** achieves the lowest error and highest R² on the held-out
  test set, and its cross-validation score is both high and stable (low std),
  suggesting it generalises well rather than overfitting to one split.
- **Random Forest** performs respectably but trails Gradient Boosting; bagging
  averages out variance but here boosting's sequential error-correction wins out.
- **Linear Regression** is the weakest of the three but still explains the
  majority of the variance, which confirms the relationship between price and
  the main features (especially Area) is close to linear at heart, with the
  tree-based models picking up the remaining non-linear location/type effects.

The best model (by test R²) is automatically selected and persisted to
`models/house_price_model.pkl` by `train_and_compare()`.

In [5]:
print('Best model:', results['best_model'])
pd.DataFrame(results['feature_importance']).head(10)

Best model: gradient_boosting


,feature,importance_pct
0,Area,66.22
1,Location_City Center,17.78
2,Location_Rural,9.36
3,Location_Suburb,2.86
4,Bedrooms,1.43
5,total_rooms,1.24
6,Age,0.72
7,bath_bed_ratio,0.21
8,area_per_room,0.14
9,Property_Type_House,0.02


## Business insights from feature importance

- **Area dominates** the prediction — by far the strongest single driver of price.
- **Location** (particularly City Center) is the second major factor, consistent
  with the price premium seen in the exploratory analysis.
- Room counts (**Bedrooms**, **Bathrooms**) and **Age** contribute meaningfully
  but far less than Area and Location.

This suggests that, from a business perspective, accurate square-footage
measurement and correct location tagging are the two most important pieces
of data quality to protect — errors there would do the most damage to
prediction accuracy.